# Task 1 합성: X종 클래스 균형화

pills/ 누끼 + backgrounds/ → X종 클래스 균형 맞춘 이미지 생성

합성 규칙: bbox 겹침 방지 · 클래스 중복 금지 · cycling pool · 각도증강 · 그림자

학습 데이터셋의 클래스 불균형을 해결하기 위해  
알약 누끼(SAM)를 추출하고 새 이미지를 생성하는 파이프라인입니다.

## 파이프라인 순서
1. 패키지 설치 및 드라이브 마운트  
2. 설정값 정의  
3. 데이터셋 로드 및 현재 출현 횟수 확인  
4. SAM으로 알약 누끼 추출 → Drive `pills/`  
5. Inpainting으로 배경 추출 → Drive `backgrounds/`  
6. 알약 + 배경 합성 → Drive `synthesized/`

## 셀 1. 임포트 및 Drive 마운트

In [ ]:
import os, json, glob, random, zipfile, io
import numpy as np
import cv2
from PIL import Image
from collections import defaultdict, Counter
from google.colab import drive
drive.mount('/content/drive')
print("Drive 마운트 완료")

## 셀 2. 경로 설정

In [ ]:
DRIVE_BASE   = "/content/drive/MyDrive"
PILLS_X_DIR  = os.path.join(DRIVE_BASE, "pills")
PILLS_N_DIR  = os.path.join(DRIVE_BASE, "pills_n18")
BG_DIR       = os.path.join(DRIVE_BASE, "backgrounds")
DATASET_DIR  = "/content/dataset"
ZIP2_PATH    = os.path.join(DRIVE_BASE, "train_56_45_merged_coco_20260703.zip")
CLEANED2_PATH= os.path.join(DRIVE_BASE, "train_56_45_cleaned.json")
CLEANED_N_PATH= os.path.join(DRIVE_BASE, "hidden_n18_cleaned.json")
ZIP_N_PATH   = os.path.join(DRIVE_BASE, "hidden_n18_aihub_train_import_20260703.zip")
SAM_CHECKPOINT = "/content/sam_vit_h_4b8939.pth"

CLUSTER = {
    "top_left"   : (230, 330, 30, 40), "top_center" : (465, 330, 30, 40),
    "top_right"  : (700, 330, 35, 40), "bot_left"   : (245, 890, 35, 60),
    "bot_center" : (465, 1000, 30, 40),"bot_right"  : (715, 840, 40, 50),
}
LAYOUT_3 = [["top_left","top_right","bot_center"],["top_center","bot_left","bot_right"]]
LAYOUT_4 = [["top_left","top_right","bot_left","bot_right"]]
SHADOW_OFFSET=(12,12); SHADOW_BLUR=18; SHADOW_OPACITY=0.45; BLEND_FEATHER=3
TARGET_COUNT=35; ROTATE_RANGE=15; RESIZE_JITTER=0.10
OVERLAP_MARGIN=10; MAX_POS_RETRIES=5; SCALE_FACTOR=0.85

for name, p in [("PILLS_X_DIR",PILLS_X_DIR),("BG_DIR",BG_DIR),
                ("ZIP2_PATH",ZIP2_PATH),("CLEANED2_PATH",CLEANED2_PATH)]:
    print(f"{name:20s}: {'OK' if os.path.exists(p) else 'MISSING'}")

TASK1_IMG = os.path.join(DRIVE_BASE, "task1_synthesized", "images")
TASK1_ANN = os.path.join(DRIVE_BASE, "task1_synthesized", "annotations")

## 셀 3. fold0에서 X종 현재 출현 횟수 계산

In [ ]:
x_current=defaultdict(int); id2name_x={}
for split in ["train","valid"]:
    jp=os.path.join(DATASET_DIR,"fold0",split,"_annotations.coco.json")
    with open(jp,encoding="utf-8") as f: coco=json.load(f)
    id2n={cat["id"]:cat["name"] for cat in coco["categories"] if cat["id"]!=0}
    id2name_x.update(id2n)
    for ann in coco["annotations"]:
        nm=id2n.get(ann["category_id"])
        if nm: x_current[nm]+=1
x_needed={n:TARGET_COUNT-c for n,c in x_current.items() if c<TARGET_COUNT}
print(f"X종 전체: {len(x_current)}개 / 추가 필요: {len(x_needed)}개 / 총 {sum(x_needed.values())}회")

## 셀 4. 배경 이미지 로드

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

bg_paths = sorted(glob.glob(os.path.join(BG_DIR, "bg_*.png")))
total    = len(bg_paths)
N_COLS   = 10
CHUNK    = 50
print(f"총 배경 이미지: {total}장")

for start in range(0, total, CHUNK):
    batch  = bg_paths[start:start+CHUNK]
    n_rows = (len(batch) + N_COLS - 1) // N_COLS
    fig, axes = plt.subplots(n_rows, N_COLS,
                              figsize=(N_COLS*2, n_rows*2))
    axes = axes.flatten()
    fig.suptitle(f"Backgrounds [{start+1}~{min(start+CHUNK,total)}/{total}]",
                 fontsize=12)

    for i, path in enumerate(batch):
        axes[i].imshow(Image.open(path).convert("RGB"))
        axes[i].set_title(os.path.basename(path), fontsize=5)
        axes[i].axis("off")

    for i in range(len(batch), len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()
    print(f"[{start+1}~{min(start+CHUNK,total)}] 완료")

## 셀 5. 합성 함수 정의

In [ ]:
def boxes_overlap(b1, b2, margin=OVERLAP_MARGIN):
    x1,y1,w1,h1=b1; x2,y2,w2,h2=b2
    return not(x1+w1+margin<x2 or x2+w2+margin<x1 or y1+h1+margin<y2 or y2+h2+margin<y1)

def any_overlap(new_bbox, placed):
    return any(boxes_overlap(new_bbox, b) for b in placed)

def create_cycling_pool(pool):
    cyc={}
    for cat,paths in pool.items():
        s=paths.copy(); random.shuffle(s)
        cyc[cat]={'paths':s,'idx':0}
    return cyc

def sample_cycling(cyc, cat):
    if isinstance(cyc, dict) and 'paths' in cyc:
        e = cyc  # 이미 entry가 넘어온 경우
    else:
        e = cyc[cat]  # 전체 pool이 넘어온 경우
    p = e['paths'][e['idx']]; e['idx'] += 1
    if e['idx'] >= len(e['paths']):
        e['idx'] = 0; random.shuffle(e['paths'])
    return p

def rotate_pill(rgba, angle):
    h,w=rgba.shape[:2]
    M=cv2.getRotationMatrix2D((w/2,h/2),angle,1.0)
    cos,sin=abs(M[0,0]),abs(M[0,1])
    nw,nh=int(h*sin+w*cos),int(h*cos+w*sin)
    M[0,2]+=(nw-w)/2; M[1,2]+=(nh-h)/2
    return cv2.warpAffine(rgba,M,(nw,nh),flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT,borderValue=(0,0,0,0))

def sample_position(key):
    cx,cy,sx,sy=CLUSTER[key]
    x=int(np.clip(np.random.normal(cx,sx),cx-2*sx,cx+2*sx))
    y=int(np.clip(np.random.normal(cy,sy),cy-2*sy,cy+2*sy))
    return x,y

def make_shadow(rgba, bg_shape, px, py):
    H,W=bg_shape[:2]; ph,pw=rgba.shape[:2]
    alpha=rgba[:,:,3].astype(np.float32)/255.0
    dx,dy=SHADOW_OFFSET; smap=np.zeros((H,W),np.float32)
    sx,sy=px+dx,py+dy
    sx1,sy1=max(0,sx),max(0,sy); sx2,sy2=min(W,sx+pw),min(H,sy+ph)
    ax1,ay1=sx1-sx,sy1-sy; ax2,ay2=ax1+(sx2-sx1),ay1+(sy2-sy1)
    if sx2>sx1 and sy2>sy1: smap[sy1:sy2,sx1:sx2]=alpha[ay1:ay2,ax1:ax2]
    k=SHADOW_BLUR*2+1; smap=cv2.GaussianBlur(smap,(k,k),SHADOW_BLUR/3)
    return smap*SHADOW_OPACITY

def paste_pill(bg_float, rgba, cx, cy, bbox_margin=10):
    H,W=bg_float.shape[:2]; ph,pw=rgba.shape[:2]
    px,py=cx-pw//2,cy-ph//2
    px1,py1=max(0,px),max(0,py); px2,py2=min(W,px+pw),min(H,py+ph)
    ax1,ay1=px1-px,py1-py; ax2,ay2=ax1+(px2-px1),ay1+(py2-py1)
    if px2<=px1 or py2<=py1: return bg_float,None
    crop=rgba[ay1:ay2,ax1:ax2]
    alpha=crop[:,:,3:4].astype(np.float32)/255.0
    rgb=cv2.cvtColor(crop,cv2.COLOR_BGRA2BGR).astype(np.float32)
    result=bg_float.copy()
    result[py1:py2,px1:px2]=rgb*alpha+result[py1:py2,px1:px2]*(1-alpha)
    ys, xs = np.where(rgba[:,:,3] > 10)
    if len(xs) > 0:
        bbox = [int(xs.min())+px - bbox_margin,
                int(ys.min())+py - bbox_margin,
                int(xs.max())-int(xs.min()) + bbox_margin*2,
                int(ys.max())-int(ys.min()) + bbox_margin*2]
    else:
        bbox = [px, py, pw, ph]
    return result, bbox

def get_pill_actual_size(rgba):
    ys,xs=np.where(rgba[:,:,3]>10)
    if len(xs)==0: return rgba.shape[1],rgba.shape[0]
    return int(xs.max()-xs.min()),int(ys.max()-ys.min())

def analyze_x_pill_sizes(pills_dir):
    widths,heights=[],[]
    for cls_dir in os.listdir(pills_dir):
        if not cls_dir.startswith("class_"): continue
        for fn in os.listdir(os.path.join(pills_dir,cls_dir)):
            if not fn.endswith(".png"): continue
            rgba=cv2.imread(os.path.join(pills_dir,cls_dir,fn),cv2.IMREAD_UNCHANGED)
            if rgba is None or rgba.shape[2]!=4: continue
            w,h=get_pill_actual_size(rgba); widths.append(w); heights.append(h)
    stats={'median_w':int(np.median(widths)),'median_h':int(np.median(heights)),
           'mean_w':int(np.mean(widths)),'mean_h':int(np.mean(heights))}
    print(f"X종 크기 분석 (n={len(widths)})")
    print(f"  너비: mean={stats['mean_w']}  median={stats['median_w']}")
    print(f"  높이: mean={stats['mean_h']}  median={stats['median_h']}")
    return stats

def resize_n_pill(rgba, tw, th, jitter=RESIZE_JITTER):
    cw,ch=get_pill_actual_size(rgba)
    if cw==0 or ch==0: return rgba
    scale=min(tw/cw,th/ch)*np.random.uniform(1-jitter,1+jitter)
    nw=max(1,int(rgba.shape[1]*scale)); nh=max(1,int(rgba.shape[0]*scale))
    return cv2.resize(rgba,(nw,nh),interpolation=cv2.INTER_LINEAR)

def build_pill_pool(pills_dir):
    pool={}
    for cls_dir in sorted(os.listdir(pills_dir)):
        if not cls_dir.startswith("class_"): continue
        cat=cls_dir[len("class_"):]
        paths=sorted([os.path.join(pills_dir,cls_dir,f)
                      for f in os.listdir(os.path.join(pills_dir,cls_dir))
                      if f.endswith(".png")])
        if paths: pool[cat]=paths
    print(f"pool: {len(pool)}개 클래스, 총 {sum(len(v) for v in pool.values())}개")
    return pool

def weighted_sample_class(needed):
    cats=list(needed.keys()); ws=[needed[c] for c in cats]
    return str(np.random.choice(cats, p=[w/sum(ws) for w in ws]))

def fill_slots(layout, combined_needed, all_pools, all_cyc, n_cats):
    used=set(); slots=[]
    for key in layout:
        avail={k:v for k,v in combined_needed.items() if k not in used}
        if avail:
            cat=weighted_sample_class(avail)
        else:
            unused=[k for k in all_pools if k not in used]
            if not unused: break
            cat=random.choice(unused)
        used.add(cat)
        is_n=cat in n_cats
        path=sample_cycling(all_cyc[cat],cat) if cat in all_cyc else random.choice(all_pools[cat])
        slots.append((key,path,cat,is_n))
    return slots

MAX_PILL_W = int(976 * 0.40)   # 390px
MAX_PILL_H = int(1280 * 0.40)  # 512px

def clip_pill_size(rgba, max_w=MAX_PILL_W, max_h=MAX_PILL_H):
    curr_w, curr_h = get_pill_actual_size(rgba)
    if curr_w <= max_w and curr_h <= max_h:
        return rgba
    scale = min(max_w/curr_w, max_h/curr_h)
    nw = max(1, int(rgba.shape[1]*scale))
    nh = max(1, int(rgba.shape[0]*scale))
    return cv2.resize(rgba, (nw, nh), interpolation=cv2.INTER_LINEAR)

def synthesize_one(bg_path, slots, cat2label, x_size_stats=None):
    bg=cv2.imread(bg_path); canvas=bg.astype(np.float32)
    ann_list=[]; placed=[]
    for key,pill_path,cat,is_n in slots:
        rgba=cv2.imread(pill_path,cv2.IMREAD_UNCHANGED)
        if rgba is None or rgba.ndim<3 or rgba.shape[2]!=4: continue
        if is_n and x_size_stats:
            rgba=resize_n_pill(rgba,x_size_stats['median_w'],x_size_stats['median_h'])
        rgba=clip_pill_size(rgba)  # 상한선 초과 시 축소
        rgba=rotate_pill(rgba,np.random.uniform(-ROTATE_RANGE,ROTATE_RANGE))
        done=False; cur=rgba.copy()
        for phase in range(2):
            if phase==1:
                h,w=cur.shape[:2]
                cur=cv2.resize(cur,(max(1,int(w*SCALE_FACTOR)),max(1,int(h*SCALE_FACTOR))),
                               interpolation=cv2.INTER_LINEAR)
            for _ in range(MAX_POS_RETRIES):
                cx,cy=sample_position(key); ph,pw=cur.shape[:2]
                est=[cx-pw//2,cy-ph//2,pw,ph]
                if not any_overlap(est,placed):
                    px,py=cx-pw//2,cy-ph//2
                    shadow=make_shadow(cur,canvas.shape,px,py)
                    canvas=canvas*(1-shadow[:,:,np.newaxis])
                    canvas,bbox=paste_pill(canvas,cur,cx,cy)
                    if bbox is not None:
                        placed.append(bbox)
                        ann_list.append({"category_id":cat2label[cat],
                                         "bbox":[float(v) for v in bbox],
                                         "area":float(bbox[2]*bbox[3]),"iscrowd":0})
                    done=True; break
            if done: break
    return canvas.astype(np.uint8),ann_list

def run_synthesis(combined_needed, all_pools, all_cyc, bg_paths, n_cats,
                  cat2label, x_size_stats, out_img, out_ann, max_images, seed):
    random.seed(seed); np.random.seed(seed)
    os.makedirs(out_img,exist_ok=True); os.makedirs(out_ann,exist_ok=True)
    imgs,anns=[],[]; img_id,ann_id=1,1
    for i in range(max_images):
        if not combined_needed:
            print(f"\n[완료] 모든 클래스 목표 달성 ({i}장)"); break
        layout=random.choice(LAYOUT_3*2+LAYOUT_4)
        slots=fill_slots(layout,combined_needed,all_pools,all_cyc,n_cats)
        if not slots: continue
        try:
            img,ann_list=synthesize_one(random.choice(bg_paths),slots,cat2label,x_size_stats)
        except Exception as e:
            print(f"\n[ERROR]: {e}"); continue
        name=f"syn_{img_id:05d}.png"
        cv2.imwrite(os.path.join(out_img,name),img)
        H,W=img.shape[:2]
        imgs.append({"id":img_id,"file_name":name,"width":W,"height":H})
        for ann in ann_list:
            ann["id"]=ann_id; ann["image_id"]=img_id; anns.append(ann); ann_id+=1
        for _,_,cat,_ in slots:
            if cat in combined_needed:
                combined_needed[cat]-=1
                if combined_needed[cat]<=0: del combined_needed[cat]
        img_id+=1
        if i%10==0: print(f"[{i+1}/{max_images}] 남은: {len(combined_needed)}개",end="\r")
    if combined_needed: print(f"\n[경고] 미달성 {len(combined_needed)}개")
    print(f"\n합성 완료: {img_id-1}장")
    cats=[{"id":0,"name":"pill","supercategory":"none"}]
    cats+=[{"id":v,"name":k,"supercategory":"pill"} for k,v in sorted(cat2label.items(),key=lambda x:x[1])]
    coco={"images":imgs,"annotations":anns,"categories":cats}
    p=os.path.join(out_ann,"_annotations.coco.json")
    with open(p,"w") as f: json.dump(coco,f,ensure_ascii=False)
    print(f"저장: {p}")
    return coco

print("함수 정의 완료")

## 셀 6. X종 cat2label 로드

In [ ]:
with open(os.path.join(DATASET_DIR,"fold0","train","_annotations.coco.json"),
          encoding="utf-8") as f:
    fold0_coco = json.load(f)

cat2label = {cat["name"]: cat["id"]
             for cat in fold0_coco["categories"] if cat["id"] != 0}
print(f"cat2label: {len(cat2label)}개 클래스")

## 셀 7. 합성 실행

> X종만 사용 (N종 없음) → N 클래스 없음

In [ ]:
os.makedirs(TASK1_ANN, exist_ok=True)
os.makedirs(TASK1_IMG, exist_ok=True)

In [ ]:
combined_needed = dict(x_needed)  # 재초기화

In [ ]:
x_pool   = build_pill_pool(PILLS_X_DIR)
x_cyc    = create_cycling_pool(x_pool)
n_cats   = set()  # Task1은 N종 없음
all_pools= dict(x_pool)
all_cyc  = dict(x_cyc)

combined_needed = dict(x_needed)  # X종 부족분
print(f"목표: {len(combined_needed)}개 클래스 / 총 {sum(combined_needed.values())}회")

task1_coco = run_synthesis(
    combined_needed = combined_needed,
    all_pools       = all_pools,
    all_cyc         = all_cyc,
    bg_paths        = bg_paths,
    n_cats          = n_cats,
    cat2label       = cat2label,
    x_size_stats    = None,   # X종만 → 리사이즈 불필요
    out_img         = TASK1_IMG,
    out_ann         = TASK1_ANN,
    max_images      = 800,
    seed            = 42,
)

## 셀 8. 균형 확인

In [ ]:
label2name = {v:k for k,v in cat2label.items()}
gen = Counter(label2name[a["category_id"]] for a in task1_coco["annotations"])
print(f"{'class':>10} {'원본':>6} {'합성':>6} {'합계':>6} {'상태':>6}")
print("-"*38)
ok=warn=0
for name in sorted(x_current.keys()):
    orig=x_current[name]; g=gen.get(name,0); total=orig+g
    flag="OK" if total>=TARGET_COUNT else "LOW"
    if total>=TARGET_COUNT: ok+=1
    else: warn+=1
    print(f"  {name:>8} {orig:>6} {g:>6} {total:>6}  {flag}")
print(f"\n정상({TARGET_COUNT}회+): {ok}개 / 미달: {warn}개")

# 셀 9. 시각화

In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict

# ── 데이터 로드 ───────────────────────────────────────────
with open(os.path.join(TASK1_ANN, "_annotations.coco.json"), encoding="utf-8") as f:
    t1_coco = json.load(f)

t1_id2img  = {img["id"]: img for img in t1_coco["images"]}
t1_id2name = {cat["id"]: cat["name"] for cat in t1_coco["categories"]}
t1_img2ann = defaultdict(list)
for ann in t1_coco["annotations"]:
    t1_img2ann[ann["image_id"]].append(ann)

# ── 시각화 ────────────────────────────────────────────────
N_COLS  = 5
CHUNK   = 50   # 한 번에 표시할 이미지 수 (메모리 고려)
img_ids = sorted(t1_id2img.keys())
total   = len(img_ids)
print(f"총 합성 이미지: {total}장")

for start in range(0, total, CHUNK):
    batch    = img_ids[start:start+CHUNK]
    n_rows   = (len(batch) + N_COLS - 1) // N_COLS
    fig, axes = plt.subplots(n_rows, N_COLS,
                              figsize=(N_COLS*3, n_rows*3))
    axes = axes.flatten()
    fig.suptitle(f"Task1 synthesized [{start+1}~{min(start+CHUNK,total)}/{total}]",
                 fontsize=12)

    for i, img_id in enumerate(batch):
        ax      = axes[i]
        img_info= t1_id2img[img_id]
        img_path= os.path.join(TASK1_IMG, img_info["file_name"])
        img     = Image.open(img_path).convert("RGB")
        ax.imshow(img)

        for ann in t1_img2ann[img_id]:
            x, y, w, h = ann["bbox"]
            cat_name    = t1_id2name.get(ann["category_id"], str(ann["category_id"]))
            rect = patches.Rectangle((x,y), w, h,
                                      linewidth=1.5, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x, y-3, cat_name, fontsize=5, color="lime",
                    bbox=dict(facecolor="black", alpha=0.4, pad=1, edgecolor="none"))

        ax.set_title(img_info["file_name"], fontsize=5)
        ax.axis("off")

    for i in range(len(batch), len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()
    print(f"[{start+1}~{min(start+CHUNK,total)}] 완료")